# **Capstone Project** # 
## Question Answering bot for [topic] based on Hybrid RAG Systems ##

## Workflow for the project ##
1. User enters the query to the bot.
2. Bot performs multi-query generation using LLM's
3. Each query is then passed onto both Semantic search and keyword search workflows.
4. The results are normalized using ranking.
5. Most appropriate results are added to the top.
6. The output is shown to the user.

In [62]:
! pip install sentence_transformers


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# load all the knowledge documents.
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader = DirectoryLoader('data/subtitles', glob="*.srt", show_progress=True, loader_cls=TextLoader)

kb_docs = loader.load()

100%|█████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 410.23it/s]


In [2]:
# Chunk the loaded documents

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = text_splitter.split_documents(kb_docs)

In [3]:
# embedd the chunks into the chroma db
import chromadb
from chromadb import Schema, SparseVectorIndexConfig, VectorIndexConfig, K
from chromadb.utils.embedding_functions import ChromaBm25EmbeddingFunction
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

# Setup API Key
f = open('keys/.chroma_api_key.txt')
CHROMA_API_KEY = f.read()

# f = open("keys/.openai_api_key.txt")
# OPENAI_API_KEY = f.read()

client = chromadb.CloudClient(
  api_key=CHROMA_API_KEY,
  tenant="a3f6801e-fc63-4a8c-88b6-e632a8d8c534",
  database='capstone_chroma_vector_db'
)

schema = Schema()
    
schema = schema.create_index(
  key='sparse_vector_key',    
  config=SparseVectorIndexConfig(
    source_key=K.DOCUMENT,
    bm25=True,
    embedding_function=ChromaBm25EmbeddingFunction(
        k=1.2,
        b=0.75,
        avg_doc_length=256.0,
        token_max_length=40
    ),
  )
)

# Configure vector index with custom embedding function
openai_ef = DefaultEmbeddingFunction()

schema = schema.create_index(
    config=VectorIndexConfig(
        space="cosine",
        embedding_function=openai_ef
    )
)

collection = client.get_or_create_collection(
  name="friends_collection",
  schema=schema,
)

collection.count()

1028

In [4]:
import uuid

# Free tier in chroma supports 300 docs to be inserted at a time
BATCH_SIZE = 250

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]

    collection.add(
        documents=[chunk.page_content for chunk in batch],
        metadatas=[chunk.metadata for chunk in batch],
        ids=[str(uuid.uuid4()) for _ in batch]
    )

    print(f"Inserted {min(i + BATCH_SIZE, len(chunks))}/{len(chunks)}")

Inserted 250/514
Inserted 500/514
Inserted 514/514


In [5]:
## searching mechanisms
from chromadb import Knn, K
from chromadb import Search
from chromadb import Rrf

def hybrid_retriever(query):
    # Sparse embeddings
    sparse_rank = Knn(
      query=query,  # Text query for sparse embeddings
      key="sparse_vector_key",  # Metadata field for sparse vectors
      return_rank=True,
      limit=20                 # Only the 20 nearest documents get scored (default limit 16)
    )
    
    # Dense semantic embeddings
    dense_rank = Knn(
      query=query,  # Text query for dense embeddings
      key="#embedding",          # Default embedding field
      return_rank=True,
      limit=20                 # Only the 20 nearest documents get scored (default limit 16)

    )
    
    # Combine with RRF
    hybrid_rank = Rrf(
      ranks=[dense_rank, sparse_rank],
      weights=[0.7, 0.3],  # 70% semantic, 30% keyword
      k=60                 # smoothing parameter - higher values reduce emphasis on top ranks
    )
    
    # Use in search
    hybrid_search = (Search()
      .rank(hybrid_rank)
      .limit(5)
      .select(K.DOCUMENT, K.SCORE, K.METADATA)
    )
    
    hybrid_results = collection.search(hybrid_search)
    
    return hybrid_results

In [6]:
# Step 3: Initialize a Chat Prompt Template

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

PROMPT_TEMPLATE = """
Answer the question based only on the following context:
{context}
Answer the question based on the above context: {question}.
Provide a detailed answer.
Don’t justify your answers.
Don’t give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
"""

prompt_template = ChatPromptTemplate(
    messages=[
        PROMPT_TEMPLATE
    ]
)

# Step 4: Initialize a Generator (i.e. Chat Model)

# Setup API Key
f = open('keys/.groq_apikey.txt')
GROQ_API_KEY = f.read()

chat_model = ChatGroq(api_key=GROQ_API_KEY, 
                           model="openai/gpt-oss-20b", 
                           temperature=0.0)

# Step 5: Initialize a Output Parser

parser = StrOutputParser()

generator_chain = prompt_template | chat_model | parser

In [7]:
# Helper function to join the retrieved chunks
retriever_documents = {}
def format_docs(docs):
    context = ""
    for row in docs.rows()[0]:
        retriever_documents[row["id"]]= row["document"]
        context += row["document"]
    return context

In [8]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Wrap the functions in RunnableWrapper
hybrid_retriever_runnable = RunnableLambda(lambda x: hybrid_retriever(x))
format_docs_runnable = RunnableLambda(lambda x: format_docs(x))

In [9]:
retriever_generator_chain = {
    "context": hybrid_retriever_runnable | format_docs_runnable, 
    "question": RunnablePassthrough()
} | generator_chain

In [10]:
# multi query generator langchain
""" Perform Multi-Query Retrieval """
query_template = """You are an AI assistant. Generate five different versions of the question:\n\nOriginal question: {question}."""
prompt_perspectives = ChatPromptTemplate.from_template(query_template)
    
multiQueryGenerator_chain = (
        prompt_perspectives | chat_model| StrOutputParser() | (lambda x: x.split("\n"))
    )

In [11]:
retrieved_docs = []

def get_unique_union(documents):
        for document in documents:
            retrieved_docs.append((userQuery, document))
        return retrieved_docs

multiQuery_retriever_chain = multiQueryGenerator_chain | retriever_generator_chain.map() | get_unique_union


In [12]:
# hugging face re ranking model.
from sentence_transformers import CrossEncoder
from operator import itemgetter

model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')

reranked_documents = {}
def reranker_output(retrieved_docs):
    scores = model.predict(retrieved_docs)
    i = 0
    for doc in retrieved_docs:
        reranked_documents[scores[i]] = doc
        i = i+1
    return build_context(reranked_documents)



def build_context(raw: dict, top_k: int = 5) -> str:
    # Normalize and sort by score descending
    items = [
        (float(score), q, a) for score, (q, a) in raw.items()
    ]
    items.sort(key=itemgetter(0), reverse=True)

    # Take top_k (optional)
    items = items[:top_k]

    # Create a readable context block
    lines = []
    for i, (score, q, a) in enumerate(items, start=1):
        lines.append(f"{i}. Score: {score:.3f}\n   Q: {q}\n   A: {a}")
    return "\n".join(lines)


In [13]:
PROMPT_TEMPLATE2 = """
Summarize the document  based only on the highest scored document:
{context}
Provide a detailed answer.
Don’t justify your answers.
Don’t give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
"""

prompt_template2 = ChatPromptTemplate(
    messages=[
        PROMPT_TEMPLATE2
    ]
)

summarizer_chain = prompt_template2 | chat_model

In [14]:
capstone_chain = multiQuery_retriever_chain| reranker_output | summarizer_chain

In [16]:
userQuery = "Who is Chandler?"
capstone_chain.invoke(userQuery)

AIMessage(content='Chandler is a person—specifically, a friend of Ross and Rachel.', additional_kwargs={'reasoning_content': 'We need to summarize the document based only on the highest scored document. Highest score is 9.164. That document: Q: Who is Chandler? A: Chandler is a person—specifically, a friend of Ross and Rachel.\n\nWe need to provide a detailed answer. But only use information from that document. So answer: Chandler is a person, friend of Ross and Rachel. Provide details? Only that. No justification. No extra info. No mention of context. Just answer.'}, response_metadata={'token_usage': {'completion_tokens': 125, 'prompt_tokens': 356, 'total_tokens': 481, 'completion_time': 0.126699573, 'completion_tokens_details': {'reasoning_tokens': 100}, 'prompt_time': 0.018486009, 'prompt_tokens_details': None, 'queue_time': 0.04384339, 'total_time': 0.145185582}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason':

In [1]:
userQuery = input('Enter your question based on Friends Series.')

Enter your question based on Friends Series. Who is Rachel


In [2]:
print(userQuery)

Who is Rachel
